In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')
df = pd.read_csv('electric_vehicle_analytics.csv', encoding='utf-8')
df.info()

: 

In [ ]:
n = len(df)
df = df.drop_duplicates()
cleared_n = len(df)
print(f'Удалено {n - cleared_n} повторяющихся записей')

In [ ]:
df = df[df['Battery_Health_%'] <= 100]
df = df[df['Battery_Health_%'] >= 0]
df = df[df['Charging_Time_hr'] >= 0.05]

In [ ]:
def hyp1_temperature_range(df):
    df['Температура'] = df['Temperature_C'].apply(
        lambda x: 'Холодно (<0°C)' if x < 0 else 'Тепло (≥0°C)'
    )
    avg_range = df.groupby('Температура')['Range_km'].mean().round(1)
    plt.figure(figsize=(9, 6))
    ax = sns.barplot(
        x=avg_range.index,
        y=avg_range.values,
        palette=["#6495ED", "#FF6347"]
    )
    for i, value in enumerate(avg_range.values):
        ax.text(i, value + 8, f'{value} км',
                ha='center', va='bottom', fontweight='bold', fontsize=14)
    plt.title('Влияние температуры эксплуатации на запас хода',
              fontsize=16, pad=20)
    plt.ylabel('Средний запас хода, км')
    plt.xlabel('Температура воздуха')
    plt.ylim(0, avg_range.max() * 1.15)
    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.show()

hyp1_temperature_range(df)

In [ ]:
def hyp2_tesla_depreciation(df):
    data = df.copy()
    data['Full_Model'] = data['Make'] + ' ' + data['Model']
    max_price = data.groupby('Full_Model')['Resale_Value_USD'].max().reset_index()
    max_price.rename(columns={'Resale_Value_USD': 'Max_Price'}, inplace=True)
    data = data.merge(max_price, on='Full_Model', how='left')
    data['Loss_%'] = (data['Max_Price'] - data['Resale_Value_USD']) / data['Max_Price'] * 100
    data['Brand'] = data['Make'].apply(lambda x: 'Tesla' if x == 'Tesla' else 'Другие марки')
    result = data.groupby('Brand')['Loss_%'].mean().round(2)
    result = result.reindex(['Tesla', 'Другие марки'])
    plt.figure(figsize=(10, 6))
    colors = ['#E31937', '#2E86AB']
    ax = sns.barplot(x=result.index, y=result.values, palette=colors)

    for i, (brand, loss) in enumerate(result.items()):
        ax.text(i, loss + 1.2, f'{loss:.1f}%',
                ha='center', va='bottom', fontsize=16, fontweight='bold', color='black')

    plt.title('Средняя потеря стоимости электромобиля при перепродаже', fontsize=14, pad=20)
    plt.ylabel('Потеря стоимости, %')
    plt.xlabel('')
    plt.ylim(0, result.max() + 10)
    plt.grid(axis='y', linestyle='--', alpha=0.3)

    ax.patches[0].set_edgecolor('white')
    ax.patches[0].set_linewidth(4)

    plt.tight_layout()
    plt.show()

hyp2_tesla_depreciation(df)

In [ ]:
def hyp3_battery_resale(df):
    data = df.copy()
    plt.figure(figsize=(11, 7))
    sns.scatterplot(data=data, x='Battery_Health_%', y='Resale_Value_USD',
                    alpha=0.65, color='#3498DB', edgecolor='white', s=70)
    sns.regplot(data=data, x='Battery_Health_%', y='Resale_Value_USD',
                scatter=False, color='#E74C3C', line_kws={'linewidth': 4})
    corr = data['Battery_Health_%'].corr(data['Resale_Value_USD']).round(3)
    plt.title(f'Зависимость остаточной стоимости от здоровья батареи\n'
              f'Коэффициент корреляции = {corr}', fontsize=16, pad=25)
    plt.xlabel('Здоровье батареи, %', fontsize=13)
    plt.ylabel('Остаточная стоимость, USD', fontsize=13)
    plt.grid(alpha=0.3, linestyle='--')

    plt.tight_layout()
    plt.show()


hyp3_battery_resale(df)

In [ ]:
def hyp4_maintenance_usage(df):
    data = df.copy()
    data['Usage_Group'] = data['Usage_Type'].replace({
        'Personal': 'Личные',
        'Fleet': 'Коммерческие',
        'Commercial': 'Коммерческие'
    })
    result = data.groupby('Usage_Group')['Maintenance_Cost_USD'].mean().round(0)
    result = result.reindex(['Личные', 'Коммерческие'])

    plt.figure(figsize=(10, 6))
    colors = ['#27AE60', '#E67E22']
    ax = sns.barplot(x=result.index, y=result.values, palette=colors)

    for i, (group, cost) in enumerate(result.items()):
        ax.text(i, cost + 40, f'${cost:,.0f}',
                ha='center', va='bottom', fontsize=16, fontweight='bold', color='black')

    plt.title('Средние затраты на обслуживание\n'
              'в зависимости от типа использования', fontsize=16, pad=20)
    plt.ylabel('Затраты на обслуживание в год, USD')
    plt.xlabel('')
    plt.ylim(0, result.max() + 400)
    plt.grid(axis='y', linestyle='--', alpha=0.3)

    plt.tight_layout()
    plt.show()


hyp4_maintenance_usage(df)

In [ ]:
def analyze_corr_matrix(df):
    data = df.copy()
    numeric_cols = data.select_dtypes(include=['float64', 'int64']).columns
    exclude_cols = ['Vehicle_ID', 'Temperature_C', 'Charging_Time_hr', 'Max_Speed_kmh']
    numeric_cols = [col for col in numeric_cols if col not in exclude_cols]
    corr_matrix = data[numeric_cols].corr()
    plt.figure(figsize=(14, 10))
    sns.heatmap(corr_matrix,
                annot=True,
                fmt='.2f',
                cmap='RdYlBu_r',
                center=0,
                linewidths=0.7,
                linecolor='white',
                cbar_kws={'shrink': 0.8},
                square=True)
    plt.title('Корреляционная матрица основных характеристик электромобилей',
              fontsize=18, pad=20)
    plt.xticks(rotation=45, ha='right', fontsize=11)
    plt.yticks(rotation=0, fontsize=11)
    plt.tight_layout()
    plt.show()

    return corr_matrix


corr = analyze_corr_matrix(df)